# 06 — Demo Interactiva · Proyecto Vena

Interfaz web con **Gradio** para demostrar el sistema en la exposicion final.
Corre directamente en Colab y genera un link publico temporal.

**Estado actual:**
- Clasificacion + Grad-CAM (NB02 v4)
- Regresion de edad (NB03) — placeholder
- Clustering (NB04) — placeholder

---

## Uso en la exposicion

1. Ejecutar todas las celdas (~2 min para descomprimir + cargar modelo)
2. Copiar el link publico que aparece al final
3. Abrir desde cualquier dispositivo
4. Usar el boton de imagen aleatoria o subir una radiografia

## Celda 1 — Montar Drive e instalar Gradio

In [1]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q gradio

print("Listo.")

Mounted at /content/drive
Listo.


## Celda 2 — Importaciones, rutas y descompresion del DatasetV2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.image import img_to_array
from PIL import Image
from pathlib import Path
import time
import gradio as gr

# ── Rutas ──────────────────────────────────────────────────────────────
RUTA_CHECKPOINTS = Path("/content/drive/MyDrive/Vena/checkpoints")
RUTA_MODELO_CLAS = RUTA_CHECKPOINTS / "mejor_modelo_clasificacion_v4_f2.keras"
RUTA_MODELO_REG  = RUTA_CHECKPOINTS / "regresion_v4.keras"  #MODIF por marvin
RUTA_MODELOS_AGR = Path("/content/drive/MyDrive/Vena/Notebooks/Apoyo/Agrupacion/modelos_produccion") # Editado por Diego
RUTA_ZIP         = Path("/content/drive/MyDrive/Vena/Notebooks/Datasets/DatasetV2.zip")
RUTA_BASE        = Path("/content/DatasetV2")


# ── Constantes ─────────────────────────────────────────────────────────
TAMANO_IMAGEN = 224
UMBRAL_CLAS   = 0.38
MARGEN_BAJA_CONFIANZA = 0.07  # si |prob - umbral| < margen, confianza baja

# Valores del StandardScaler del NB03
EDAD_MEDIA = 50.3
EDAD_STD   = 17.1
PESO_META  = 10.0

# ── Descomprimir DatasetV2 ─────────────────────────────────────────────
csvs = ["master_train.csv", "master_val.csv", "master_test.csv"]
ya_existe = all((RUTA_BASE / c).exists() for c in csvs)

if not ya_existe:
    assert RUTA_ZIP.exists(), f"No se encontro: {RUTA_ZIP}"
    print("Descomprimiendo DatasetV2...")
    inicio = time.time()
    !unzip -q -o "{RUTA_ZIP}" -d "/content/"
    if not (RUTA_BASE / "master_test.csv").exists():
        sub = RUTA_BASE / "DatasetV2"
        if (sub / "master_test.csv").exists():
            RUTA_BASE = sub
    print(f"Listo en {time.time() - inicio:.0f}s")
else:
    if not (RUTA_BASE / "master_test.csv").exists():
        sub = RUTA_BASE / "DatasetV2"
        if (sub / "master_test.csv").exists():
            RUTA_BASE = sub
    print("DatasetV2 ya descomprimido.")

print(f"TensorFlow : {tf.__version__}")
print(f"Gradio     : {gr.__version__}")

Descomprimiendo DatasetV2...
Listo en 41s
TensorFlow : 2.20.0
Gradio     : 5.50.0


## Celda 3 — Cargar modelo, test set y capa Grad-CAM

In [3]:
# ── Modelo ─────────────────────────────────────────────────────────────
modelo_clas = load_model(str(RUTA_MODELO_CLAS))
print(f"Modelo: {RUTA_MODELO_CLAS.name}")

# ── Capa Grad-CAM ──────────────────────────────────────────────────────
nombre_capa_conv = None
for capa in reversed(modelo_clas.layers):
    try:
        forma = tuple(capa.output.shape)
    except (AttributeError, RuntimeError):
        continue
    if len(forma) == 4 and forma[1] is not None and forma[1] > 1:
        nombre_capa_conv = capa.name
        break

print(f"Capa Grad-CAM: {nombre_capa_conv}")

# ── Test set ───────────────────────────────────────────────────────────
df_test = pd.read_csv(RUTA_BASE / "master_test.csv")
df_test["label"] = df_test["label"].astype(int)
df_test["ruta_completa"] = df_test["path"].apply(lambda p: str(RUTA_BASE / p))
print(f"Test set: {len(df_test):,} imagenes")

#agregado por Marvin
modelo_reg = load_model(str(RUTA_MODELO_REG), compile= False)
print(f"Regresión: {RUTA_MODELO_REG.name}")

# Constantes del StandardScaler (de NB02_Regresion_v4)
EDAD_MEDIA = 50.3
EDAD_STD   = 17.1

# ~~~ Agrupacion ~~~
import joblib, json

# Modelos de agrupación
scaler_cluster = joblib.load(RUTA_MODELOS_AGR / "v_scaler.joblib")        # ← NUEVO
pca_cluster    = joblib.load(RUTA_MODELOS_AGR / "v_pca_reducer.joblib")   # ← NUEVO
kmeans_cluster = joblib.load(RUTA_MODELOS_AGR / "v_kmeans_model.joblib")  # ← NUEVO

# Metricas de agrupacion
PERFILES_CLUSTER = {
    0: {"estado": "Mixto (54.6% sanos)",     "genero": "Femenino (98.0%)",  "edad_media": 48.7},
    1: {"estado": "Enfermos (88.2%)",         "genero": "Masculino (73.3%)", "edad_media": 51.0},
    2: {"estado": "Sanos (67.9%)",            "genero": "Mixto (52.4% F)",   "edad_media": 36.7},
    3: {"estado": "Enfermos (93.8%)",         "genero": "Mixto (60.0% M)",   "edad_media": 64.2},
    4: {"estado": "Mixto (53.3% enfermos)",   "genero": "Masculino (96.1%)", "edad_media": 52.9},
    5: {"estado": "Sanos (64.6%)",            "genero": "Masculino (96.2%)", "edad_media": 49.3},
}

# ← NUEVO: cargar edad_min/edad_max del entrenamiento
with open(RUTA_MODELOS_AGR / "v_meta.json") as f:
    _meta = json.load(f)
EDAD_MIN_CLUSTER = _meta["edad_min"]
EDAD_MAX_CLUSTER = _meta["edad_max"]

print("Modelos de agrupación cargados.")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 176 variables whereas the saved optimizer has 180 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Modelo: mejor_modelo_clasificacion_v4_f2.keras
Capa Grad-CAM: relu
Test set: 14,684 imagenes
Regresión: regresion_v4.keras


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MiniBatchKMeans from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid re

Modelos de agrupación cargados.


## Celda 4 — Funciones del pipeline

In [4]:
def generar_gradcam(modelo, imagen_array, nombre_capa):
    """Genera mapa de calor Grad-CAM."""
    capa_conv   = modelo.get_layer(nombre_capa)
    grad_modelo = Model(
        inputs  = modelo.input,
        outputs = [capa_conv.output, modelo.output]
    )
    with tf.GradientTape() as tape:
        salida_conv, prediccion = grad_modelo(imagen_array)
        prob = prediccion[0, 0]

    gradientes = tape.gradient(prob, salida_conv)
    pesos = tf.reduce_mean(gradientes, axis=(0, 1, 2))
    mapa = salida_conv[0] @ pesos[..., tf.newaxis]
    mapa = tf.squeeze(mapa)
    mapa = tf.nn.relu(mapa)
    maximo = tf.reduce_max(mapa)
    if maximo > 0:
        mapa = mapa / maximo
    return mapa.numpy(), float(prob)


def superponer_heatmap(imagen, heatmap, alfa=0.4):
    """Superpone Grad-CAM sobre la imagen."""
    heatmap_grande = tf.image.resize(
        heatmap[..., np.newaxis], (imagen.shape[0], imagen.shape[1]),
        method="bicubic"
    ).numpy().squeeze()
    heatmap_grande = np.clip(heatmap_grande, 0, 1)
    colores = cm.get_cmap("jet")(heatmap_grande)[:, :, :3]
    superposicion = (1 - alfa) * imagen + alfa * colores
    return np.clip(superposicion, 0, 1)


def interpretar_confianza(prob):
    """
    Genera texto de diagnostico y confianza.
    Si la probabilidad esta cerca del umbral, advierte confianza baja.
    """
    es_anomalia = prob >= UMBRAL_CLAS
    distancia_al_umbral = abs(prob - UMBRAL_CLAS)
    confianza_baja = distancia_al_umbral < MARGEN_BAJA_CONFIANZA

    if es_anomalia:
        diagnostico = "ANOMALIA DETECTADA"
        confianza_pct = prob
    else:
        diagnostico = "Sin hallazgos significativos"
        confianza_pct = 1 - prob

    texto = f"{confianza_pct:.0%}  (p = {prob:.3f})"
    if confianza_baja:
        texto += "\n** Resultado cercano al umbral de decision."
        texto += "\n** Se recomienda revision por especialista."

    return diagnostico, texto


def analizar_radiografia(imagen_pil, patient_age, patient_gender):   # ← NUEVO: 2 params extra
    if imagen_pil is None:
        return None, "", "", "", ""

    img_resized = imagen_pil.resize((TAMANO_IMAGEN, TAMANO_IMAGEN))
    img_array   = img_to_array(img_resized) / 255.0
    img_batch   = np.expand_dims(img_array, axis=0)

    # Clasificación
    prob = float(modelo_clas.predict(img_batch, verbose=0)[0, 0])
    diagnostico, texto_confianza = interpretar_confianza(prob)

    # Grad-CAM
    heatmap, _ = generar_gradcam(modelo_clas, img_batch, nombre_capa_conv)
    img_gradcam = superponer_heatmap(img_array, heatmap, alfa=0.45)
    img_gradcam_uint8 = (img_gradcam * 255).astype(np.uint8)

    # Regresión de edad
    pred_norm  = float(modelo_reg.predict(img_batch, verbose=0)[0, 0])
    edad_pred  = pred_norm * EDAD_STD + EDAD_MEDIA
    edad_texto = f"{edad_pred:.0f} años (±6 años aprox.)"

    # ── NUEVO: Clustering ─────────────────────────────────────────────
    try:
        # 1. Extraer embedding usando el backbone del modelo de clasificación
        #    (GlobalAveragePooling2D ya está en el modelo — última capa antes del Dense)
        capa_gap = None
        for capa in reversed(modelo_clas.layers):
            if "global_average" in capa.name.lower():
                capa_gap = capa.name
                break

        extractor_emb = Model(
            inputs  = modelo_clas.input,
            outputs = modelo_clas.get_layer(capa_gap).output
        )
        embedding = extractor_emb.predict(img_batch, verbose=0)  # (1, 1024)

        # 2. Normalizar edad con los rangos del entrenamiento
        edad_norm  = (patient_age - EDAD_MIN_CLUSTER) / (EDAD_MAX_CLUSTER - EDAD_MIN_CLUSTER)
        edad_norm  = float(np.clip(edad_norm, 0, 1))

        # 3. Codificar género: M=0, F=1
        genero_cod = 1.0 if str(patient_gender).upper() == "F" else 0.0

        # 4. Escalar + concatenar metadatos (igual que NB03 CELL 15)
        emb_scaled = scaler_cluster.transform(embedding)
        X = np.hstack([
            emb_scaled,
            np.array([[edad_norm  * PESO_META]]),
            np.array([[genero_cod * PESO_META]])
        ])

        # 5. PCA → K-Means
        X_pca      = pca_cluster.transform(X)
        cluster_id = int(kmeans_cluster.predict(X_pca)[0])
        perfil = PERFILES_CLUSTER[cluster_id]
        cluster_texto = (
            f"Grupo {cluster_id}\n"
            f"Estado   : {perfil['estado']}\n"
            f"Género   : {perfil['genero']}\n"
            f"Edad media: {perfil['edad_media']} años"
        )

    except Exception as e:
        cluster_texto = f"Error: {e}"
    # ──────────────────────────────────────────────────────────────────

    return img_gradcam_uint8, diagnostico, texto_confianza, edad_texto, cluster_texto

def imagen_aleatoria():
    """Selecciona una imagen aleatoria del test set."""
    fila = df_test.sample(n=1).iloc[0]
    img_pil = Image.open(fila["ruta_completa"]).convert("RGB")
    etiqueta_real = "Anomalia" if fila["label"] == 1 else "Normal"

    edad   = int(fila["patient_age"]) if pd.notna(fila.get("patient_age")) else 50
    genero = fila["patient_gender"].strip().upper() if pd.notna(fila.get("patient_gender")) else "M"
    hallazgo = fila["original_finding"] if pd.notna(fila.get("original_finding")) else "No Finding"

    info = f"Real: {etiqueta_real}  |  Edad: {edad} años  |  Género: {genero}  |  Hallazgo: {hallazgo}"

    return img_pil, info, edad, genero


print("Funciones definidas.")


Funciones definidas.


## Celda 5 — Interfaz Gradio

Layout de 3 columnas para aprovechar el ancho en pantallas grandes:
1. Entrada (imagen + botones)
2. Resultados (diagnostico + metricas)
3. Grad-CAM (mapa de calor)

In [ ]:
# ── Cargar logo ────────────────────────────────────────────────────────
RUTA_LOGO = Path("/content/drive/MyDrive/Vena/Notebooks/Apoyo/logo_vena.png")
logo_pil = Image.open(str(RUTA_LOGO)).convert("RGBA") if RUTA_LOGO.exists() else None

# ── CSS personalizado ──────────────────────────────────────────────────
# Gradio inyecta este string como una hoja de estilos dentro del <head>
# de la página. Cada regla afecta la interfaz entera.
CSS = """
/* Fondo negro en todo el contenedor y en el body */
body, .gradio-container {
    background-color: #000000 !important;
}

/* Fondo carmesí en el texto de cada etiqueta de componente.
   .block label span es el selector donde Gradio 5 pone ese texto. */
.block label span {
    background-color: #DC143C !important;   /* rojo carmesí */
    color: #ffffff !important;
    padding: 2px 8px;
    border-radius: 4px;
    font-weight: 600;
}

/* Texto general en blanco para que se vea sobre el fondo negro */
.gradio-container, .gradio-container * {
    color: #ffffff;
}

/* Bordes de los componentes en gris oscuro */
.block {
    border-color: #333333 !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="VENA - Interpretando el tórax humano",
    css=CSS                        # ← aquí se inyecta el CSS
) as demo:

    # ── Header: logo + título ─────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=1, min_width=120):
            if logo_pil is not None:
                gr.Image(
                    value=logo_pil,
                    interactive=False,
                    show_label=False,
                    height=100,
                    container=False   # sin caja alrededor
                )
        with gr.Column(scale=4):
            gr.Markdown(
                "# VENA\n"
                "### Interpretando el tórax humano\n"
                "*Samsung Innovation Campus - Proyecto Final*\n\n"
                "---"
            )

    # ── Estado interno (edad y género vienen de imagen_aleatoria) ─────
    state_edad   = gr.State(value=50)
    state_genero = gr.State(value="M")

    with gr.Row(equal_height=True):

        # ── Columna 1: Entrada ────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### Entrada")
            img_entrada = gr.Image(
                type="pil",
                label="Radiografía de tórax",
                height=280
            )
            txt_info = gr.Textbox(
                label="Información de la imagen",
                interactive=False,
                lines=1
            )
            btn_aleatorio = gr.Button(
                "Imagen aleatoria del test set",
                variant="secondary"
            )
            btn_analizar = gr.Button(
                "Analizar radiografía",
                variant="primary",
                size="lg"
            )

        # ── Columna 2: Resultados ─────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### Resultado")
            txt_resultado = gr.Textbox(
                label="Diagnóstico",          # ← quitado "(NB02)"
                interactive=False,
                lines=1
            )
            txt_confianza = gr.Textbox(
                label="Confianza del modelo",
                interactive=False,
                lines=3
            )
            txt_edad = gr.Textbox(
                label="Edad estimada",         # ← quitado "(NB03)"
                interactive=False,
                lines=1
            )
            txt_cluster = gr.Textbox(
                label="Grupo de paciente",     # ← quitado "(NB04)"
                interactive=False,
                lines=4
            )
            gr.Markdown(
                "*Herramienta de apoyo clínico.* "
                "*No reemplaza el diagnóstico médico.*"
            )

        # ── Columna 3: Grad-CAM ───────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### Zonas de atención (Grad-CAM)")
            img_gradcam = gr.Image(
                label="Mapa de calor superpuesto",
                height=350
            )
            gr.Markdown(
                "Rojo = alta atención del modelo\n\n"
                "Azul = baja atención"
            )

    # ── Conectar botones ──────────────────────────────────────────────
    btn_aleatorio.click(
        fn      = imagen_aleatoria,
        inputs  = [],
        outputs = [img_entrada, txt_info, state_edad, state_genero]
    )
    btn_analizar.click(
        fn      = analizar_radiografia,
        inputs  = [img_entrada, state_edad, state_genero],
        outputs = [img_gradcam, txt_resultado, txt_confianza, txt_edad, txt_cluster]
    )

    gr.Markdown(
        "---\n"
        "**Modelo:** DenseNet121 fine-tuneado | "
        "**Dataset:** NIH + CheXpert (~96k imgs) | "
        "**AUC:** 0.833 | "
        "**Umbral:** 0.38"
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_9056/2783529217.py:35: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_9056/2783529217.py:35: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2121796668ca7aa2d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_9056/3276521352.py:30: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colores = cm.get_cmap("jet")(heatmap_grande)[:, :, :3]


---

## Guia para integrar regresion y clustering

Solo hay que modificar la funcion `analizar_radiografia()` en la Celda 4.
Los campos en la interfaz ya existen.

### Regresion (NB03)

```python
# En Celda 3, agregar:
RUTA_MODELO_REG = RUTA_CHECKPOINTS / "regresion_v3.h5"
modelo_reg = load_model(str(RUTA_MODELO_REG), compile=False)

# En analizar_radiografia(), reemplazar el placeholder:
pred_norm = float(modelo_reg.predict(img_batch, verbose=0)[0, 0])
edad_pred = pred_norm * EDAD_STD + EDAD_MEDIA
edad_texto = f"{edad_pred:.0f} anios (error tipico: +/- 6 anios)"
```

### Clustering (NB04)

```python
# Extraer embedding y predecir cluster
import joblib
kmeans = joblib.load(str(RUTA_CHECKPOINTS / "kmeans.pkl"))
# ... extraer embedding con backbone, predecir cluster
cluster_texto = f"Grupo {cluster_id}"
```